In [1]:
# 0_Data_Labeling.ipynb
import os
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime, timedelta
from tqdm import tqdm  # 進度條 (pip install tqdm)

# --- 設定 ---
BASE_DIR = r"C:\Users\User-NB\OneDrive\Desktop\ML"
IN_CSV = os.path.join(BASE_DIR, "Fundamentals Annual.csv") 
OUT_CSV = os.path.join(BASE_DIR, "Fundamentals Annual_withY.csv")

# ★★★ 請在此填入您的 Email (SEC API 強制要求 User-Agent) ★★★
SEC_USER_AGENT = "changtruda@gmail.com"

# 1. 讀取原始財報資料
print(f"正在讀取原始檔案: {IN_CSV}")
df = pd.read_csv(IN_CSV)

# 2. 確保有 CIK 欄位 (SEC 查詢需要)
# 如果 WRDS 下載的資料有 cik，通常是 float，要轉成 10位數補0的字串
if 'cik' not in df.columns:
    raise ValueError("錯誤：資料集中找不到 'cik' 欄位。請重新從 WRDS 下載並包含 CIK，或使用 Ticker 轉換。")

df['cik'] = df['cik'].fillna(0).astype(int).astype(str).str.zfill(10)
df['datadate'] = pd.to_datetime(df['datadate']) # 確保財報日期是時間格式

# 3. 定義 SEC 爬蟲函數
def get_sec_delisting_dates(cik_list):
    """
    輸入 CIK 列表，回傳一個字典 {cik: [list of Form 25 filing dates]}
    """
    headers = {'User-Agent': SEC_USER_AGENT}
    delisting_map = {}
    
    print(f"開始爬取 {len(cik_list)} 家公司的 SEC Form 25 資料...")
    print("注意：SEC 限制每秒 10 次請求，請耐心等待...")

    for cik in tqdm(cik_list):
        url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        try:
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                filings = data.get('filings', {}).get('recent', {})
                
                if filings:
                    forms = filings.get('form', [])
                    dates = filings.get('filingDate', [])
                    
                    # 篩選出 Form 25 相關的申報 (25, 25-NSE 等)
                    f25_dates = []
                    for form, date in zip(forms, dates):
                        if form.startswith('25'): # 抓取 25 或 25-NSE
                            f25_dates.append(pd.to_datetime(date))
                    
                    if f25_dates:
                        delisting_map[cik] = sorted(f25_dates)
            
            # SEC Rate Limit 限制 (安全起見，每次停 0.11 秒)
            time.sleep(0.11)
            
        except Exception as e:
            print(f"Error fetching CIK {cik}: {e}")
            
    return delisting_map

# 4. 執行爬蟲 (只針對不重複的 CIK 爬一次，節省時間)
unique_ciks = df['cik'].unique()

# 如果你之前已經爬過存成暫存檔，可以加一段讀取邏輯，這裡直接跑
print("準備連線 SEC EDGAR...")
delisting_dict = get_sec_delisting_dates(unique_ciks)

print(f"爬取完成。共有 {len(delisting_dict)} 家公司有 Form 25 紀錄。")

# 5. 標記 Y 標籤
# 邏輯：如果在財報日期 (datadate) 後的 400 天內，有 Form 25 紀錄，則 Y=1
print("正在標記 Y (Delisting Prediction)...")

def check_delisting(row):
    cik = row['cik']
    report_date = row['datadate']
    
    if cik in delisting_dict:
        # 檢查該公司的所有 Form 25 日期
        for d_date in delisting_dict[cik]:
            # 預警窗口：財報日 < 下市日 <= 財報日 + 400天
            if report_date < d_date <= (report_date + timedelta(days=400)):
                return 1
    return 0

# 使用 apply 進行標記 (速度可能稍慢，但邏輯最清楚)
df['Y'] = df.apply(check_delisting, axis=1)

# 6. 後續處理 (補值與格式化)
df['gvkey'] = df['gvkey'].astype(str)
df['fyear'] = pd.to_numeric(df['fyear'], errors='coerce').fillna(0).astype(int)

# 7. 數據驗證
print(f"\n【數據驗證】")
print(f"公司總數 (GVKEY): {df['gvkey'].nunique()} ")
print(f"總樣本數 (Rows): {len(df)}")
print(f"Form 25 違約樣本數 (Y=1): {df['Y'].sum()}")
print(f"違約比例: {df['Y'].mean():.4f} ")

if df['Y'].sum() == 0:
    print("警告：沒有標記到任何 Y=1，請檢查 User-Agent 或日期窗口設定。")

# 8. 存檔
df.to_csv(OUT_CSV, index=False)
print(f"\n已儲存包含 SEC 標籤的檔案至: {OUT_CSV}")
print("請繼續執行下一個檔案：1_EDA_data_prep.ipynb")

正在讀取原始檔案: C:\Users\User-NB\OneDrive\Desktop\ML\Fundamentals Annual.csv
準備連線 SEC EDGAR...
開始爬取 9551 家公司的 SEC Form 25 資料...
注意：SEC 限制每秒 10 次請求，請耐心等待...


100%|██████████| 9551/9551 [1:22:40<00:00,  1.93it/s]


爬取完成。共有 3670 家公司有 Form 25 紀錄。
正在標記 Y (Delisting Prediction)...

【數據驗證】
公司總數 (GVKEY): 16237 
總樣本數 (Rows): 79844
Form 25 違約樣本數 (Y=1): 3728
違約比例: 0.0467 

已儲存包含 SEC 標籤的檔案至: C:\Users\User-NB\OneDrive\Desktop\ML\Fundamentals Annual_withY.csv
請繼續執行下一個檔案：1_EDA_data_prep.ipynb


In [2]:
import pandas as pd
import os
BASE_DIR = r"C:\Users\User-NB\OneDrive\Desktop\ML"
RAW_CSV = os.path.join(BASE_DIR, "Fundamentals Annual_withY.csv")
df = pd.read_csv(RAW_CSV)
print("CSV 欄位名稱:", df.columns.tolist())

CSV 欄位名稱: ['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'gvkey', 'datadate', 'cik', 'sic', 'fyear', 'act', 'at', 'che', 'dlc', 'dltt', 'invt', 'lct', 'lt', 'rect', 'seq', 'cogs', 'dp', 'ebit', 'ni', 'oibdp', 'sale', 'txt', 'xint', 'xsga', 'oancf', 'csho', 'prcc_f', 'Y']
